In [1]:
import pandas as pd

df = pd.read_excel("../datasets/hospital_dataset.xlsx")
df.head(10)

,review,sentiment
0,istri saya sissandra arisi dirawat setelah ope...,positive
1,"tq dr titos, dr suskhan dan suster pelayanan y...",positive
2,suster isti di bagian pendaftaran poliklinik n...,positive
3,"thank you utk dr. titos , dr. tjien ronny & al...",positive
4,terimaksih kepada dokter fery darmawan spog yg...,positive
5,pelayanan rs pik dari segi kecepatan penangana...,positive
6,saya sakit ，di rawat di 7603 dokter dan suster...,positive
7,rs pik selalu jd rs ternyaman kl lagi sakit 😁 ...,positive
8,terima kasih kepada doktef tjen ronny dan para...,positive
9,"slma bbrp hari dirawat dikamar 7512 puas bgt, ...",positive


## Text Preprocessing Pipeline

Setup text preprocessing using indoNLP library

In [2]:
from indoNLP.preprocessing import pipeline, emoji_to_words, replace_word_elongation
from indoNLP.preprocessing import SLANG_DATA, SLANG_PATTERN
import re

# Create a safer version of replace_slang that handles missing keys
def safe_replace_slang(text):
    """Replace slang words, but keep original if not found in dictionary"""
    return re.sub(SLANG_PATTERN, lambda mo: SLANG_DATA.get(mo.group(0).lower(), mo.group(0)), text)

# Use safe_replace_slang instead of the library's version
pipe = pipeline([safe_replace_slang, emoji_to_words, replace_word_elongation])

print("✓ Text preprocessing pipeline ready")

✓ Text preprocessing pipeline ready


## Data Cleaning

Remove duplicates and missing values

In [3]:
# Check data before cleaning
print(f"Original data shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

# Remove rows with missing values
df = df.dropna()

# Remove duplicate rows
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print(f"\nAfter cleaning:")
print(f"Cleaned data shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

Original data shape: (25295, 2)
Missing values:
review       0
sentiment    0
dtype: int64
Duplicate rows: 1375

After cleaning:
Cleaned data shape: (23920, 2)
Missing values: 0
Duplicate rows: 0


## Create Labels and Final Dataset

Create labels based on score and prepare `source_final.csv`

In [4]:
df['label'] = df['score'].apply(lambda x: 'positive' if x > 3 else 'negative')

df_final = df[['content', 'label', 'score']].copy()

# Clean df_final: remove NA and duplicates
print(f'df_final before cleaning: {len(df_final)} rows')
print(f"Missing values:\n{df_final.isnull().sum()}")
print(f"Duplicate rows: {df_final.duplicated().sum()}")

df_final = df_final.dropna()
df_final = df_final.drop_duplicates()
df_final = df_final.reset_index(drop=True)

print(f'\ndf_final after cleaning: {len(df_final)} rows')
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicate rows: {df_final.duplicated().sum()}")

# Display label distribution
print(f'\nLabel distribution:')
print(df_final['label'].value_counts())
print(f'\nFirst 10 rows:')
df_final.head(10)

KeyError: 'score'

## Apply Text Preprocessing

Clean and normalize text content

In [ ]:
print("Applying text preprocessing to 'content' column...")
print(f"Total rows to process: {len(df_final)}")

# Apply preprocessing pipeline
df_final['content'] = df_final['content'].apply(pipe)

print("✓ Text preprocessing complete!")
print("\nSample of preprocessed content:")
df_final.head(10)

Applying text preprocessing to 'content' column...
Total rows to process: 141002
✓ Text preprocessing complete!

Sample of preprocessed content:


,content,label,score
0,akun gopay saya di blok,negative,1
1,Lambat sekali sekarang ini bosssku apk gojek e...,negative,3
2,Kenapa sih dari kemarin saya buka aplikasi goj...,positive,4
3,Baru download gojek dan hape baru terus ditop ...,negative,1
4,Mantap,positive,5
5,Bagus,positive,4
6,Coba dulu,negative,2
7,Ok,positive,5
8,bagaimana ini kak pin saya salah terus padahal...,negative,1
9,Biar aman kamu tidak bisa pakai gojek Jadi say...,negative,1


In [ ]:
df_final.to_csv('../source_final.csv', index=False)
print('Successfully created source_final.csv!')

Successfully created source_final.csv!


## Create Balanced Dataset

Create `source_balanced.csv` with 30,000 positive and 30,000 negative samples

In [ ]:
# Check current label distribution
print("Current label distribution:")
print(df_final['label'].value_counts())
print(f"\nTotal samples: {len(df_final)}")

# Sample 30k positive and 30k negative
positive_samples = df_final[df_final['label'] == 'positive'].sample(n=30000, random_state=42)
negative_samples = df_final[df_final['label'] == 'negative'].sample(n=30000, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([positive_samples, negative_samples], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset:")
print(f"Total rows: {len(df_balanced)}")
print(f"\nLabel distribution:")
print(df_balanced['label'].value_counts())
df_balanced.head(10)

Current label distribution:
label
positive    79447
negative    61555
Name: count, dtype: int64

Total samples: 141002

Balanced dataset:
Total rows: 60000

Label distribution:
label
positive    30000
negative    30000
Name: count, dtype: int64


,content,label,score
0,Pengiriman cepat banget,positive,5
1,Jalan masuk ke dalam perumahan bellacasa depok...,negative,1
2,Kenapa apl gojeku sering banget kaya enggak bi...,negative,2
3,Aplikasinya sangat membantu sekali dan gampang...,positive,5
4,Alhamdulillah sellu mendapatkan driver yang ba...,positive,5
5,makin lama makin serakah aplikasi anak bangsa ...,negative,1
6,Keren banyak diskon,positive,5
7,Bagus amat deh ini,positive,5
8,enggak bisa didownload,positive,5
9,aturan promo nya berubah dibeda2in di hp orang...,negative,1


In [ ]:
# Save balanced dataset to CSV
df_balanced.to_csv('../source_balanced.csv', index=False)